<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/VIX_PRODUCTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX Production v1 — inférence légère, à relancer régulièrement

**Rôle.** Contrairement à tous les notebooks précédents (qui **valident** une méthodologie en
walk-forward, donc entraînent toujours sur un passé strictement antérieur au test), celui-ci sert
au **déploiement** : il gèle la config gagnante établie par ce projet — **RandomForest, régime
GLOBAL, horizon 5 jours, N=8 features sélectionnées par SHAP, rééquilibrage SMOTE** (F1_dir≈0.610,
F1_UP_FORT≈0.359, F1_DOWN_FORT≈0.627 en walk-forward) — et l'entraîne sur **tout** l'historique
disponible, pour produire une prédiction sur la ligne la plus récente.

**Ce n'est plus un test de méthodologie, c'est un modèle qu'on utilise.** D'où la différence
volontaire avec les notebooks de validation : ici, entraîner sur 100% des données passées est
correct et souhaitable (on ne "triche" pas — il n'y a pas de test à protéger, juste une prédiction
sur l'inconnu de demain).

**Prérequis** : `VIX_FINAL_FEATURES.ipynb` doit avoir tourné (au moins une fois) et poussé son
dataset. **À relancer périodiquement** (par exemple chaque semaine) après avoir relancé
`VIX_FINAL_FEATURES` pour rafraîchir les données — le modèle est ré-entraîné à chaque exécution,
il n'y a pas d'état à faire persister d'une exécution à l'autre à part le **journal des
prédictions**, poussé sur `results/vix-production`, qui s'enrichit d'une ligne à chaque run et
permet de suivre a posteriori comment les prédictions ont évolué dans le temps.

**Limite assumée** : la fraîcheur de la prédiction dépend de la fraîcheur du dataset partagé
(donc du dernier run de `VIX_FINAL_FEATURES`), pas d'un téléchargement live à chaque exécution —
un choix délibéré pour garder ce notebook simple, l'horizon de prédiction étant de toute façon de
5 jours (un décalage de quelques jours sur les données sources n'invalide pas l'exercice).


In [1]:
import subprocess, sys
pkgs = ['xgboost', 'shap', 'imbalanced-learn', 'pyarrow', 'joblib']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


Installation OK


In [2]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import shap
import joblib
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_PRODUCTION'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'flat_thr': 0.003,
    'horizon': 5,
    'N': 8,
    'sampler': 'SMOTE',
    'algo': 'RandomForest',
    'shap_sample': 500,
    'pool_prefilter': 450,
}
TARGET_COL = 'VIX_Amplitude_Class'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
RESULTS_BRANCH = 'results/vix-production'
LOG_CSV = 'vix_production_log.csv'
MODEL_FILE = 'vix_production_model.joblib'

print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | config gelée : h={CONFIG['horizon']}j GLOBAL "
      f"N={CONFIG['N']} {CONFIG['sampler']} {CONFIG['algo']} | référence walk-forward : "
      f"F1_dir=0.610±0.025  F1_UP_FORT=0.359  F1_DOWN_FORT=0.627")


VIX_PRODUCTION v1 | config gelée : h=5j GLOBAL N=8 SMOTE RandomForest | référence walk-forward : F1_dir=0.610±0.025  F1_UP_FORT=0.359  F1_DOWN_FORT=0.627


In [3]:
# ============================================================
# CHARGEMENT DU DATASET PARTAGÉ (produit par VIX_FINAL_FEATURES)
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not os.path.exists('vix_final_features.parquet'):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = "/content/_vix_features_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", FEATURES_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(
            "Impossible de récupérer le dataset partagé depuis "
            f"'{FEATURES_BRANCH}'. As-tu bien exécuté VIX_FINAL_FEATURES.ipynb au moins une fois "
            f"(et poussé son résultat) ? Détail: {clone.stderr[-500:]}")
    subprocess.run(["cp", f"{workdir}/vix_final_features.parquet", "."], check=True)
    subprocess.run(["cp", f"{workdir}/vix_final_features_meta.json", "."], check=True)
    print(f"[PULL OK] Dataset récupéré depuis '{FEATURES_BRANCH}'")
else:
    print("[SKIP] vix_final_features.parquet déjà présent localement")

df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
FEATURE_POOL = meta['feature_pool']
VIX_COL = meta['vix_col']; SPX_COL = meta['spx_col']
all_dates = df_features.dropna(how='all').index.sort_values()
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | pool: {len(FEATURE_POOL)} features | "
      f"source: {meta['date_min']} → {meta['date_max']} (dernière ligne disponible : {all_dates[-1].date()})")


[PULL OK] Dataset récupéré depuis 'results/vix-final-features'
Dataset: (6908, 1300) | VIX=IDX_VIX | pool: 1102 features | source: 2000-01-03 → 2026-07-21 (dernière ligne disponible : 2026-07-21)


In [4]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def get_clf(algo):
    if algo == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1)
    raise ValueError(algo)

def get_samp(name):
    return {'SMOTE': SMOTE(random_state=SEED)}[name]

def shap_rank(X_tr, y_tr, pool_names, top_n, prefilter):
    nf = X_tr.shape[1]
    if nf > prefilter:
        pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                           eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
        pf.fit(X_tr, y_tr); keep = np.argsort(pf.feature_importances_)[::-1][:prefilter]
    else:
        keep = np.arange(nf)
    Xk = X_tr[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y_tr)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    order = np.argsort(np.asarray(arr).ravel())[::-1][:top_n]
    return list(keep[order])

print("Helpers OK (build_target, get_clf, get_samp, shap_rank)")


Helpers OK (build_target, get_clf, get_samp, shap_rank)


## Principe : validation vs déploiement

Tout ce projet, jusqu'ici, a cherché à **estimer honnêtement** la performance d'une config
(modèle + features + horizon) sur des données qu'elle n'a jamais vues — d'où le soin extrême
apporté au walk-forward, au purging, à l'absence de fuite temporelle. Cette estimation
(F1_dir≈0.610) reste valable une fois qu'on a décidé de **déployer** cette config, mais la
question change : il ne s'agit plus d'évaluer, mais de produire la meilleure prédiction possible
pour une date qui n'existe pas encore dans l'historique.

**Conséquence** : en déploiement, on entraîne sur 100% des données disponibles (au lieu de
réserver un test), parce qu'il n'y a plus rien à protéger — chaque jour de donnée supplémentaire
ne peut qu'améliorer l'estimation des seuils de régime, la sélection SHAP et l'ajustement du
modèle. C'est l'inverse de la logique de validation, et c'est voulu : un modèle de production
qui n'utiliserait pas les données les plus récentes serait moins bon sans raison.

**Ce que ce notebook NE fait PAS** : il ne ré-estime pas la performance attendue (ça a déjà été
fait, de façon rigoureuse, dans les notebooks de validation) — le F1_dir≈0.610 affiché en
référence est celui obtenu en walk-forward sur `VIX_FINAL_ML_SCAN`/`VIX_CHAMPION_WF`, pas un
recalcul ici. Le modèle entraîné sur 100% des données sera nécessairement optimiste s'il était
réévalué sur son propre train — ce biais est normal et attendu en production, tant qu'on ne
confond jamais ce chiffre avec une nouvelle estimation de performance.


In [5]:
# ============================================================
# ENTRAÎNEMENT FINAL SUR 100% DE L'HISTORIQUE DISPONIBLE
# ============================================================
target_full, reg_r_full, thr_full = build_target(df_features[VIX_COL], CONFIG['horizon'], len(all_dates))
idx = target_full.index
X_pool_all = df_features[FEATURE_POOL].reindex(idx)

sc = RobustScaler()
Xs = sc.fit_transform(np.nan_to_num(X_pool_all.values))
y = target_full.values.astype(int)

fidx = shap_rank(Xs, y, FEATURE_POOL, CONFIG['N'], CONFIG['pool_prefilter'])
selected_features = [FEATURE_POOL[i] for i in fidx]
print(f"Features retenues (N={CONFIG['N']}) : {selected_features}")

try:
    Xr, yr = get_samp(CONFIG['sampler']).fit_resample(Xs[:, fidx], y)
except Exception:
    Xr, yr = Xs[:, fidx], y

clf = get_clf(CONFIG['algo'])
clf.fit(Xr, yr)
print(f"Modèle final entraîné sur {len(y)} lignes ({y.min()}..{y.max()} classes) "
      f"→ {len(yr)} lignes après {CONFIG['sampler']}.")

joblib.dump({'model': clf, 'scaler': sc, 'selected_features': selected_features,
             'feature_indices': fidx, 'config': CONFIG, 'regime_thresholds': thr_full,
             'trained_through': str(all_dates[-1].date())}, MODEL_FILE)
print(f"[SAVE] {MODEL_FILE}")


Features retenues (N=8) : ['FRED_NFCI_ret_5d', 'IDX_VIX_zscore_60d', 'vix_vs_ma10__zrel__SBUX_ret_5d', 'vix_level', 'VZ_vol_20d', 'spike_vix_over_3m', 'IDX_VIX_zscore_60d__minus__HON_zscore_60d', 'vix_zscore_10d__zrel__EWC_zscore_60d']
Modèle final entraîné sur 6768 lignes (0..3 classes) → 7672 lignes après SMOTE.
[SAVE] vix_production_model.joblib


In [6]:
# ============================================================
# PRÉDICTION SUR LA DERNIÈRE LIGNE DISPONIBLE (AUJOURD'HUI)
# ============================================================
CLASS_NAMES = {0: 'DOWN_FORT', 1: 'DOWN_faible', 2: 'UP_faible', 3: 'UP_FORT'}

last_date = all_dates[-1]
X_latest_raw = df_features[FEATURE_POOL].iloc[[-1]].values
X_latest = sc.transform(np.nan_to_num(X_latest_raw))[:, fidx]
proba = clf.predict_proba(X_latest)[0]
pred_class = int(np.argmax(proba))

vix_now = df_features[VIX_COL].ffill().iloc[-1]
calm_thr, stress_thr = thr_full.get('GLOBAL', (None, None))
vix_all = df_features[VIX_COL].ffill()
calm_q, stress_q = vix_all.quantile(0.33), vix_all.quantile(0.67)
regime_now = 'CALM' if vix_now < calm_q else ('STRESS' if vix_now >= stress_q else 'NORMAL')

print(f"=== Prédiction VIX à {CONFIG['horizon']}j — {last_date.date()} ===")
print(f"VIX actuel : {vix_now:.2f} | régime courant : {regime_now}")
print(f"Classe prédite : {pred_class} ({CLASS_NAMES[pred_class]})")
print("Probabilités par classe :")
for c, name in CLASS_NAMES.items():
    print(f"  {name:12s} : {proba[c]:.1%}")
direction = 'UP' if pred_class in (2, 3) else 'DOWN'
amplitude = 'FORT' if pred_class in (0, 3) else 'faible'
print(f"\nRésumé : le modèle anticipe une variation **{direction}** du VIX à {CONFIG['horizon']} jours, "
      f"d'amplitude **{amplitude}** (confiance {proba[pred_class]:.1%}).")
print("\n[RAPPEL] Performance attendue (walk-forward, pas recalculée ici) : "
      "F1_dir=0.610±0.025  F1_UP_FORT=0.359  F1_DOWN_FORT=0.627 — à utiliser comme repère de "
      "fiabilité, pas comme garantie sur cette prédiction individuelle.")

pred_row = {'run_date': pd.Timestamp.now().strftime('%Y-%m-%d'), 'data_through': str(last_date.date()),
            'vix_now': round(float(vix_now), 4), 'regime_now': regime_now,
            'pred_class': pred_class, 'pred_label': CLASS_NAMES[pred_class],
            'direction': direction, 'amplitude': amplitude,
            'proba_DOWN_FORT': round(float(proba[0]), 4), 'proba_DOWN_faible': round(float(proba[1]), 4),
            'proba_UP_faible': round(float(proba[2]), 4), 'proba_UP_FORT': round(float(proba[3]), 4)}


=== Prédiction VIX à 5j — 2026-07-21 ===
VIX actuel : 17.35 | régime courant : NORMAL
Classe prédite : 1 (DOWN_faible)
Probabilités par classe :
  DOWN_FORT    : 25.2%
  DOWN_faible  : 28.1%
  UP_faible    : 20.5%
  UP_FORT      : 26.2%

Résumé : le modèle anticipe une variation **DOWN** du VIX à 5 jours, d'amplitude **faible** (confiance 28.1%).

[RAPPEL] Performance attendue (walk-forward, pas recalculée ici) : F1_dir=0.610±0.025  F1_UP_FORT=0.359  F1_DOWN_FORT=0.627 — à utiliser comme repère de fiabilité, pas comme garantie sur cette prédiction individuelle.


In [7]:
# ============================================================
# JOURNALISATION DE LA PRÉDICTION + PUSH DU MODÈLE (results/vix-production)
# ============================================================
_PUSH_WORKDIR = "/content/_vix_production_push"

def push_production():
    if not GITHUB_TOKEN:
        print("[SKIP] Pas de GITHUB_TOKEN — la prédiction ci-dessus reste valable localement.")
        return
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0:
            print(f"[WARN] clone: {clone.stderr[-300:]}"); return
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)

        log_path = f"{_PUSH_WORKDIR}/{LOG_CSV}"
        if os.path.exists(log_path):
            df_log = pd.read_csv(log_path)
            df_log = df_log[df_log['run_date'] != pred_row['run_date']]
            df_log = pd.concat([df_log, pd.DataFrame([pred_row])], ignore_index=True)
        else:
            df_log = pd.DataFrame([pred_row])
        df_log.to_csv(log_path, index=False)

        subprocess.run(["cp", MODEL_FILE, f"{_PUSH_WORKDIR}/{MODEL_FILE}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email", "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name", "VIX Production Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", LOG_CSV, MODEL_FILE], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Prédiction production {pred_row['run_date']} — {pred_row['pred_label']}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] Modèle + journal des prédictions sur '{RESULTS_BRANCH}' "
                  f"({len(df_log)} prédictions journalisées au total)")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_production()


[PUSH OK] Modèle + journal des prédictions sur 'results/vix-production' (1 prédictions journalisées au total)
